# Bridge & User-Defined Networks

Notebook 12 introduced all five network drivers. This notebook goes deep on the one you'll use every day: the user-defined bridge. It covers subnet configuration, Docker's embedded DNS, network aliases, connecting and disconnecting containers at runtime, internal networks, and the patterns that underpin real multi-container architectures.

Topics:
- Creating and configuring bridge networks (subnet, gateway, IP range)
- Docker's embedded DNS server
- Network aliases — multiple names for the same container
- Connecting and disconnecting containers at runtime
- `--internal` networks — no external routing
- Assigning static IPs
- Network-based service isolation patterns
- Inspecting live connectivity

## 1. Creating Bridge Networks

```bash
docker network create [OPTIONS] NETWORK_NAME
```

With no options, Docker picks a subnet from the `172.x.0.0/16` range automatically. You can override everything:

```bash
docker network create \
    --driver bridge \
    --subnet 10.10.0.0/24 \
    --gateway 10.10.0.1 \
    --ip-range 10.10.0.128/25 \
    --label env=production \
    prod-net
```

| Option | Meaning |
|--------|--------|
| `--subnet` | CIDR range for the network |
| `--gateway` | Gateway IP (usually the first usable host) |
| `--ip-range` | Sub-range within `--subnet` from which Docker allocates container IPs |
| `--label` | Metadata, same as container labels |
| `--internal` | No outbound routing — containers can't reach the internet |
| `--attachable` | Allow standalone containers to attach (required in Swarm context) |

In [ ]:
import subprocess, json

# Create a network with an explicit subnet
!docker network create \
    --subnet 10.10.0.0/24 \
    --gateway 10.10.0.1 \
    appnet

# Inspect it
raw = subprocess.check_output(["docker", "network", "inspect", "appnet"])
data = json.loads(raw)[0]
ipam = data['IPAM']['Config'][0]

print(f"Driver:  {data['Driver']}")
print(f"Subnet:  {ipam['Subnet']}")
print(f"Gateway: {ipam['Gateway']}")
print(f"Internal: {data['Internal']}")

## 2. Docker's Embedded DNS

Every user-defined bridge network has an embedded DNS server at `127.0.0.11`. Containers on the network get this address as their nameserver automatically — you never need to configure it.

When container A queries `http://api`, Docker DNS:
1. Checks if `api` is a container name on the same network
2. Checks if `api` is a network alias on the same network
3. Falls back to the host's DNS for external names

```
Container: curl http://api/users
               │
               ▼ DNS query: api
         127.0.0.11 (Docker DNS)
               │
               ├─ Found: container named 'api' → 10.10.0.3
               │   or
               └─ Not found → forward to host DNS (8.8.8.8, etc.)
```

This is why user-defined bridges are so powerful for microservices — services discover each other by name, and the IP can change (e.g. after a container restart) without any reconfiguration.

In [ ]:
import time

# Start two services on the same network
!docker run -d --name api     --network appnet nginx:alpine
!docker run -d --name frontend --network appnet alpine sleep 120

time.sleep(1)

# frontend resolves 'api' by name via Docker DNS
print("DNS resolution from 'frontend':")
!docker exec frontend nslookup api 2>/dev/null

# Check nameserver configuration inside the container
print("\nNameserver configured in /etc/resolv.conf:")
!docker exec frontend cat /etc/resolv.conf

In [ ]:
# External names also resolve — Docker DNS falls through to host DNS
print("External DNS (falls through to host):")
!docker exec frontend nslookup example.com 2>/dev/null | head -6

## 3. Network Aliases

A **network alias** is an additional hostname for a container on a specific network. Multiple containers can share the same alias — Docker DNS returns all their IPs, enabling round-robin load balancing.

```bash
docker run --network appnet --network-alias backend myapp
```

Use cases:
- **Blue/green deployment** — old and new versions both have the alias; traffic shifts as you add/remove containers
- **Load distribution** — multiple instances behind one alias
- **Decoupling** — consumers use the alias, not the container name; you can rename containers without updating consumers

In [ ]:
# Two containers share the alias 'db' — Docker DNS returns both IPs
!docker run -d --name db-primary \
    --network appnet \
    --network-alias db \
    alpine sleep 120

!docker run -d --name db-replica \
    --network appnet \
    --network-alias db \
    alpine sleep 120

time.sleep(1)

# DNS returns multiple A records for the alias 'db'
print("DNS records for alias 'db' (both containers):")
!docker exec frontend nslookup db 2>/dev/null | grep -E 'Name|Address'

# Each lookup may return a different order (round-robin)
print("\nRound-robin: repeated lookups return different orderings:")
for i in range(3):
    result = subprocess.check_output(
        ["docker", "exec", "frontend", "nslookup", "db"],
        stderr=subprocess.DEVNULL
    ).decode()
    addrs = [l.split()[-1] for l in result.splitlines() if 'Address' in l and '127.0.0.11' not in l]
    print(f"  Lookup {i+1}: {addrs}")

## 4. Connecting and Disconnecting at Runtime

You can add or remove a container from a network without stopping it:

```bash
docker network connect    NETWORK CONTAINER  # add container to network
docker network disconnect NETWORK CONTAINER  # remove container from network
```

This is how you attach a container to multiple networks — the container gets a separate IP on each.

In [ ]:
# Create a second network and connect 'api' to both
!docker network create adminnet

# api starts on appnet; connect it to adminnet too
!docker network connect adminnet api

# api now has two IP addresses — one per network
raw = subprocess.check_output(["docker", "inspect", "api",
                               "--format", "{{json .NetworkSettings.Networks}}"])
nets = json.loads(raw)
print("Networks 'api' is connected to:")
for name, cfg in nets.items():
    print(f"  {name}: {cfg['IPAddress']}")

In [ ]:
# Disconnect api from adminnet — immediately takes effect
!docker network disconnect adminnet api

raw = subprocess.check_output(["docker", "inspect", "api",
                               "--format", "{{json .NetworkSettings.Networks}}"])
nets = json.loads(raw)
print("Networks after disconnect:")
for name, cfg in nets.items():
    print(f"  {name}: {cfg['IPAddress']}")

!docker network rm adminnet

## 5. Static IP Assignment

You can pin a container to a specific IP within a network's subnet:

```bash
docker run --network appnet --ip 10.10.0.50 mycontainer
```

Static IPs require a user-defined network with an explicit subnet — you can't use static IPs on the default bridge. The IP must be within the network's subnet and not already in use.

**When to use static IPs:** Rarely. Container names and aliases are almost always a better solution because they survive container restarts. Static IPs are useful when integrating with external systems that expect a fixed IP (legacy firewalls, hardcoded config in systems you don't control).

In [ ]:
# Assign a static IP
!docker run -d --name static-ip-demo \
    --network appnet \
    --ip 10.10.0.99 \
    alpine sleep 60

!docker inspect static-ip-demo \
    --format '{{.NetworkSettings.Networks.appnet.IPAddress}}'

!docker rm -f static-ip-demo

## 6. Internal Networks — No External Routing

An `--internal` network has no gateway to the outside world. Containers can talk to each other on the network but cannot reach the internet or the host.

```bash
docker network create --internal private-backend
```

This is a strong isolation primitive — even if a container in the internal network is compromised, it cannot make outbound connections.

In [ ]:
!docker network create --internal internal-net

!docker run -d --name internal-a --network internal-net alpine sleep 60
!docker run -d --name internal-b --network internal-net alpine sleep 60

time.sleep(1)

# Can reach each other
print("internal-a → internal-b (same network):")
!docker exec internal-a ping -c 2 internal-b 2>&1 | tail -2

# Cannot reach the internet
print("\ninternal-a → internet (blocked):")
!docker exec internal-a ping -c 1 -W 2 8.8.8.8 2>&1 | tail -2 || echo "No route — internal network"

!docker rm -f internal-a internal-b
!docker network rm internal-net

## 7. Network-Based Service Isolation Patterns

In a real multi-service application, you use multiple networks to enforce isolation — each tier of the architecture can only reach the tiers it needs to.

```
internet
    │
    │  -p 80:80
    ▼
[nginx proxy]  ─── frontend-net ───► [api service]
                                            │
                                     backend-net (internal)
                                            │
                                     ┌──────┴──────┐
                                 [postgres]     [redis]
```

- `nginx` is on `frontend-net` and publishes port 80 to the host
- `api` is on both `frontend-net` (receives requests from nginx) and `backend-net` (reaches databases)
- `postgres` and `redis` are on `backend-net` only — unreachable from the internet or the proxy
- `backend-net` is `--internal` — databases cannot make outbound connections

In [ ]:
# Build the isolation pattern
!docker network create frontend-net
!docker network create --internal backend-net

# nginx proxy — frontend-net only, publishes port 80
!docker run -d --name proxy --network frontend-net -p 8180:80 nginx:alpine

# api — on both networks (the bridge between frontend and backend)
!docker run -d --name api-svc --network frontend-net nginx:alpine
!docker network connect backend-net api-svc

# database — backend-net only, no direct internet access
!docker run -d --name db-svc --network backend-net alpine sleep 120

time.sleep(1)

# Verify isolation
print("proxy → api-svc (frontend-net): ", end="")
result = subprocess.run(["docker", "exec", "proxy", "wget", "-qO-", "http://api-svc"],
                        capture_output=True, timeout=5)
print("OK" if result.returncode == 0 else "FAIL")

print("proxy → db-svc (different network): ", end="")
result = subprocess.run(["docker", "exec", "proxy", "ping", "-c", "1", "-W", "2", "db-svc"],
                        capture_output=True, timeout=5)
print("BLOCKED" if result.returncode != 0 else "reachable")

print("api-svc → db-svc (backend-net): ", end="")
result = subprocess.run(["docker", "exec", "api-svc", "ping", "-c", "1", "db-svc"],
                        capture_output=True, timeout=5)
print("OK" if result.returncode == 0 else "FAIL")

print("db-svc → internet (internal net): ", end="")
result = subprocess.run(["docker", "exec", "db-svc", "ping", "-c", "1", "-W", "2", "8.8.8.8"],
                        capture_output=True, timeout=5)
print("BLOCKED" if result.returncode != 0 else "reachable")

In [ ]:
# Clean up everything
!docker rm -f proxy api-svc db-svc api frontend
!docker rm -f db-primary db-replica static-ip-demo 2>/dev/null || true
!docker network rm appnet frontend-net backend-net 2>/dev/null || true
print("Cleaned up")

## 8. Useful Network Commands Reference

```bash
# Create
docker network create mynet
docker network create --subnet 10.0.0.0/24 --gateway 10.0.0.1 mynet
docker network create --internal secure-net

# List and inspect
docker network ls
docker network inspect mynet
docker network inspect mynet --format '{{range .Containers}}{{.Name}} {{.IPv4Address}}{{"\n"}}{{end}}'

# Connect / disconnect running containers
docker network connect    mynet mycontainer
docker network connect    mynet mycontainer --alias backend
docker network disconnect mynet mycontainer

# Remove
docker network rm mynet
docker network prune        # remove all unused networks

# Find what network a container is on
docker inspect mycontainer --format '{{json .NetworkSettings.Networks}}'

# Find all containers on a network
docker network inspect mynet --format '{{range .Containers}}{{.Name}}  {{end}}'
```

## Summary

- **User-defined bridge** is the right default for multi-container apps — create one per logical group of services.
- **Subnet/gateway/IP-range** can all be customised at creation time. Docker auto-assigns from the `172.x.x.x` range if not specified.
- **Embedded DNS at `127.0.0.11`** resolves container names and aliases automatically — no `/etc/hosts` hacking needed.
- **Network aliases** (`--network-alias`) let multiple containers share a hostname — Docker DNS returns all IPs (round-robin). Essential for scaling and blue/green deployments.
- **`docker network connect` / `disconnect`** attaches or detaches a running container from a network with immediate effect.
- **Static IPs** (`--ip`) pin a container to a specific address — use sparingly; names and aliases are almost always better.
- **`--internal` networks** block all outbound routing — containers on the network can only reach each other. Strong isolation primitive for databases and internal services.
- **Multi-network architecture** — put each tier on a separate network; connect the service that bridges tiers (e.g. the API) to both. Databases sit on `--internal` networks and are never directly reachable from the proxy or the internet.